# Chapter 5 — Fundamentals of Machine Learning

Maps to Chollet Ch.5. Ch.4 built models; this chapter is **why they work and how to make them work
better**. The single most useful chapter for squeezing out contest accuracy.

### The one tension behind everything
- **Optimization** = doing well on *training* data (you control this).
- **Generalization** = doing well on *unseen* data (you can't control it directly — only fit training data).
- Push optimization too hard → **overfitting** → generalization drops.

Training goes: **underfit** (room to improve, train & val both fall) → **robust fit** (the sweet spot) →
**overfit** (train keeps falling, val turns *up*). Your job: stop at the sweet spot.

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras
from keras import layers
import numpy as np, matplotlib.pyplot as plt

# small MNIST subset -> overfits fast, demos run in seconds
(Xtr_full, ytr_full), (Xte, yte) = keras.datasets.mnist.load_data()
Xtr_full = Xtr_full.reshape(-1, 784).astype("float32")/255
Xte      = Xte.reshape(-1, 784).astype("float32")/255
X = Xtr_full[:2000]; y = ytr_full[:2000]          # tiny train set on purpose
print("toy train:", X.shape)


## 1. A net can memorize *anything* (so low training loss proves nothing)
Shuffle the labels so inputs and targets are unrelated. Training loss still drops (the model memorizes),
but **validation never improves** — there's nothing to generalize to. Lesson: only the *validation* curve
tells you if real learning is happening.

In [ ]:
y_random = y.copy(); np.random.shuffle(y_random)   # destroy the input->label relationship

def small_model():
    m = keras.Sequential([layers.Dense(512, activation="relu"),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

h = small_model().fit(X, y_random, epochs=20, batch_size=128,
                      validation_split=0.2, verbose=0)
print(f"final TRAIN acc on random labels: {h.history['accuracy'][-1]:.2f}  (memorized)")
print(f"final VAL   acc on random labels: {h.history['val_accuracy'][-1]:.2f}  (~chance 0.1)")


## 2. What causes overfitting
- **Noisy / mislabeled data** — model contorts to fit outliers.
- **Ambiguous regions** — same input maps to different labels (inherent uncertainty).
- **Rare features & spurious correlations** — a feature appearing in a few samples gets a huge weight
  by coincidence. *This is the most common real cause.*

Demo: append 784 **pure-noise** columns to MNIST. Same real information, but the model latches onto
random correlations in the noise → validation accuracy drops vs. appending zeros.

In [ ]:
Xn = np.concatenate([X, np.random.random((len(X), 784))], axis=1)   # +noise features
Xz = np.concatenate([X, np.zeros((len(X), 784))],          axis=1)   # +zero features

def model_1568():
    m = keras.Sequential([layers.Dense(256, activation="relu"),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

hn = model_1568().fit(Xn, y, epochs=15, batch_size=128, validation_split=0.2, verbose=0)
hz = model_1568().fit(Xz, y, epochs=15, batch_size=128, validation_split=0.2, verbose=0)
print(f"val acc with NOISE channels: {max(hn.history['val_accuracy']):.3f}")
print(f"val acc with ZERO  channels: {max(hz.history['val_accuracy']):.3f}  (higher)")


### Manifold hypothesis (why DL generalizes at all)
Real data (digits, faces, speech) lives on a **low-dimensional manifold** inside the huge input space.
A deep net is a smooth curve fitted by gradient descent; partway through training it approximates that
manifold and can **interpolate** between training points → that's generalization. Consequences:
- More/denser/cleaner **data** is the #1 lever (interpolation needs dense sampling).
- Noisy/irrelevant features hurt — they distort the manifold.

## 3. Evaluating models *honestly*
Split into **train / validation / test**:
- **train**: fit weights. **val**: tune hyperparameters (layers, units, epochs). **test**: touch *once*, at the very end.
- **Information leak**: every decision made from the val score leaks val info into the model. Tune enough and
  you overfit the *validation* set — so the final, unbiased number must come from the untouched **test** set.

**Protocols (pick by data size):**
| protocol | when |
|---|---|
| hold-out (single split) | lots of data |
| **K-fold** CV | small/medium data, noisy scores |
| iterated K-fold (shuffle + repeat) | very little data, need precision (Kaggle) |

**Always set a common-sense baseline first.** Majority-class accuracy, or random (0.1 for 10-class MNIST,
0.5 for balanced binary). If you can't beat it, the model is worthless.

In [ ]:
from sklearn.model_selection import KFold
def kfold_score(build_fn, X, y, k=4, epochs=15):
    scores=[]
    for tr, va in KFold(k, shuffle=True, random_state=42).split(X):
        m = build_fn()
        m.fit(X[tr], y[tr], epochs=epochs, batch_size=128, verbose=0)
        scores.append(m.evaluate(X[va], y[va], verbose=0, return_dict=True)["accuracy"])
    return np.mean(scores), np.std(scores)

# common-sense baseline = always predict the majority class
from collections import Counter
maj = Counter(y).most_common(1)[0][0]
print(f"baseline (majority-class) acc: {(y==maj).mean():.3f}")
mean, std = kfold_score(small_model, X, y, k=4)
print(f"4-fold model acc: {mean:.3f} ± {std:.3f}   (must beat baseline)")


## 4. Underfitting? Improve the *fit* first
If train loss won't go down / stalls high:
- **Learning rate** — too high diverges, too low crawls. Try ×10 / ÷10. (Most common fix.)
- **Capacity** — add layers/units so the model *can* represent the pattern.
- **Train longer** — more epochs.
Get the model to comfortably overfit a small batch first; that proves the pipeline can learn.

## 5. Overfitting? Improve *generalization* — the toolkit
In rough order of impact:
1. **Get more / cleaner data**, **feature engineering**, **feature selection** (drop noise features).
2. **Reduce capacity** — smaller model can't memorize as much.
3. **Weight regularization (L2)** — penalize large weights → smoother curve. `kernel_regularizer=l2(1e-3)`.
4. **Dropout** — randomly zero a fraction of activations during training → prevents co-adaptation.
5. **Early stopping** — stop at the val-loss minimum (free, always use it).

### Showdown: one overfitting baseline vs. four fixes (watch the val-loss curves)

In [ ]:
from keras import regularizers

def make(kind):
    if kind == "baseline":      # big model, no regularization -> overfits
        lays = [layers.Dense(512, activation="relu"),
                layers.Dense(512, activation="relu"),
                layers.Dense(10, activation="softmax")]
    elif kind == "smaller":
        lays = [layers.Dense(32, activation="relu"),
                layers.Dense(10, activation="softmax")]
    elif kind == "L2":
        r = regularizers.l2(1e-3)
        lays = [layers.Dense(512, activation="relu", kernel_regularizer=r),
                layers.Dense(512, activation="relu", kernel_regularizer=r),
                layers.Dense(10, activation="softmax")]
    elif kind == "dropout":
        lays = [layers.Dense(512, activation="relu"), layers.Dropout(0.5),
                layers.Dense(512, activation="relu"), layers.Dropout(0.5),
                layers.Dense(10, activation="softmax")]
    m = keras.Sequential(lays)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

hist = {}
for kind in ["baseline", "smaller", "L2", "dropout"]:
    hist[kind] = make(kind).fit(X, y, epochs=40, batch_size=128,
                                validation_split=0.3, verbose=0).history


In [ ]:
plt.figure(figsize=(7,4))
for kind, h in hist.items():
    plt.plot(h["val_loss"], label=kind, lw=2 if kind!="baseline" else 2.5,
             ls="--" if kind=="baseline" else "-")
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend()
plt.title("baseline overfits (val loss ↑); regularizers keep it down"); plt.show()
for kind, h in hist.items():
    print(f"{kind:9s} best val_loss = {min(h['val_loss']):.3f}  | best val_acc = {max(h['val_accuracy']):.3f}")


### Early stopping — stop automatically at the val minimum (combine with the above)

In [ ]:
cb = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
h = make("dropout").fit(X, y, epochs=200, batch_size=128, validation_split=0.3,
                        callbacks=[cb], verbose=0)
print(f"stopped after {len(h.history['loss'])} epochs (asked for 200)")
print(f"restored best val_acc = {max(h.history['val_accuracy']):.3f}")


---
# ✍️ PROBLEMS

### P1 — Find the overfitting point
Train the `baseline` model for 60 epochs on the toy set, plot train vs val **loss** on one axis. Mark the
epoch where val loss is minimal. How many epochs *past* that does train loss keep dropping? That gap is
pure overfitting.

In [ ]:
# TODO


### P2 — Regularization strength sweep
For L2, sweep `l2 ∈ {0, 1e-4, 1e-3, 1e-2}`; for dropout, sweep `rate ∈ {0, 0.2, 0.5, 0.7}`. Plot best
val accuracy vs strength for each. Where's the sweet spot? What happens when you over-regularize?

In [ ]:
# TODO


### P3 — Honest evaluation on a small set
Take 300 MNIST samples. (a) Report accuracy from a single 80/20 hold-out, repeated with 5 different random
seeds — how much does it swing? (b) Now report 5-fold CV mean±std. Which is more trustworthy and why?

In [ ]:
# TODO


### P4 — Beat the baseline on diabetes
On `diabetes.csv` (Ch.4 loader), compute the majority-class baseline, then build a *regularized* MLP
(dropout + early stopping) and beat it. Report 5-fold CV mean±std and confirm it clears the baseline by a
clear margin.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Regularized model (L2 + dropout)

In [ ]:
from keras import layers, regularizers
import keras
model = keras.Sequential([
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-3)),
    layers.Dropout(0.5),
    layers.Dense(64,  activation="relu", kernel_regularizer=regularizers.l2(1e-3)),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])


### T2 — Early stopping + checkpoint (always use)

In [ ]:
cbs = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best.keras", save_best_only=True, monitor="val_loss"),
]
hist = model.fit(X_train, y_train, validation_split=0.2,
                 epochs=200, batch_size=64, callbacks=cbs, verbose=0)


### T3 — Common-sense baseline (do this BEFORE modeling)

In [ ]:
from collections import Counter
import numpy as np
maj = Counter(y_train).most_common(1)[0][0]
print("majority-class baseline:", (y_test == maj).mean())          # classification
# regression baseline: predict the mean
# from sklearn.metrics import mean_absolute_error
# print(mean_absolute_error(y_test, np.full_like(y_test, y_train.mean())))


### T4 — K-fold CV (reliable score on small data)

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
def build_model():
    ...  # return a fresh COMPILED model
scores=[]
for tr, va in KFold(5, shuffle=True, random_state=42).split(X):
    m = build_model()
    m.fit(X[tr], y[tr], epochs=40, batch_size=64, verbose=0)
    scores.append(m.evaluate(X[va], y[va], verbose=0, return_dict=True)["accuracy"])
print(f"{np.mean(scores):.3f} ± {np.std(scores):.3f}")


---
### ✅ Checklist
- [ ] Explain optimization vs generalization and the underfit→robust→overfit arc.
- [ ] Know the 3 overfitting causes; explain why low training loss alone means nothing.
- [ ] Set up train/val/test correctly and explain the information-leak / why test is touched once.
- [ ] Choose hold-out vs K-fold by data size; always set a common-sense baseline.
- [ ] Diagnose underfit (fix lr/capacity/epochs) vs overfit (data/capacity/L2/dropout/early stopping).
- [ ] Run the regularization showdown and read the val-loss curves.

**Next: Chapter 6** — the *universal workflow of ML*: a repeatable end-to-end checklist for attacking any
new problem (define task → develop → deploy). Then Ch.7 Keras deep-dive, Ch.8 CNNs (+ CNN-from-scratch).
Say "Chapter 6".